**CatBoost (short for Categorical Boosting)** is a fast, open-source machine learning algorithm developed by Yandex that uses gradient boosting on decision trees

# Core Intuition


1. **Model Initialization**

    Set baseline guess and global prior using the target mean:
    $$\hat{y}^{(0)} = \bar{y} = p$$
    
2. **Boosting Round $m$**
    
    **Step A: Ordered Target Statistics (Categorical Encoding)**

    Calculate categorical features sequentially under permutation $\sigma$ to prevent target leakage:
    $$\hat{x}_{i, \text{cat}} = \frac{\sum_{j < i, \text{cat}_j = \text{cat}_i} y_j + (a \cdot p)}{\sum_{j < i, \text{cat}_j = \text{cat}_i} 1 + a}$$
    
    **Step B: Ordered Boosting Gradients & Hessians**

    Evaluate error using auxiliary sub-model $M_{i-1}$ trained strictly on preceding samples ($j < i$):
    - Gradient: $g_i = M_{i-1}(x_i) - y_i$
    - Hessian: $h_i = 1.0$
    
    **Step C: Symmetric Tree Split Finding & Leaf Weights**

    Evaluate **identical split rules** (e.g. for both nodes split may be x> 2.5) across all nodes at the same tree depth:
    
    - **Similarity Score:**
    $$\text{Sim} = \frac{\left(\sum g_i\right)^2}{\sum h_i + \lambda}$$
    
    - **Gain:**
    $$\text{Gain} = \text{Sim}_L + \text{Sim}_R - \text{Sim}_{\text{Parent}}$$
    
    - **Optimal Leaf Weight:**
    $$w = -\frac{\sum g_i}{\sum h_i + \lambda}$$
    
    **Step D: Update Prediction**
    Update predictions for all samples ($i = 1 \dots N$):
    $$\hat{y}_i^{(m)} = \hat{y}_i^{(m-1)} + \eta \cdot w(x_i)$$
    
3. **Final Prediction**
$$\hat{y}_{\text{final}} = \bar{y} + \eta \cdot \sum_{m=1}^M w_m(x)$$

# Mathematical Example


**Dataset** ($N=4$):

- Sample 1: $x_1 = (\text{"City\_A"}, 1.0)$, $y_1 = 10$
- Sample 2: $x_2 = (\text{"City\_B"}, 2.0)$, $y_2 = 20$
- Sample 3: $x_3 = (\text{"City\_A"}, 5.0)$, $y_3 = 40$
- Sample 4: $x_4 = (\text{"City\_B"}, 6.0)$, $y_4 = 50$

**Hyperparameters**: $\eta = 0.5$, $\lambda = 1.0$, $a = 1.0$ (smoothing weight)


1. **Base Model Initialization (m=0)**

    $$\hat{y}_i^{(0)} = \bar{y} = \frac{10 + 20 + 40 + 50}{4} = \mathbf{30.0}$$
    Global prior $p = 30.0$.



2. **CatBoost Ordered Target Statistics (Categorical Encoding)**

    Using fixed permutation order $\sigma = (1, 2, 3, 4)$:
    $$\hat{x}_{i, \text{cat}} = \frac{\sum_{j < i, \text{cat}_j = \text{cat}_i} y_j + (a \cdot p)}{\sum_{j < i, \text{cat}_j = \text{cat}_i} 1 + a}$$

    - Sample 1 ($\text{"City\_A"}$): $\hat{x}_{1, \text{cat}} = \frac{0 + (1.0 \cdot 30.0)}{0 + 1.0} = \mathbf{30.0}$
    - Sample 2 ($\text{"City\_B"}$): $\hat{x}_{2, \text{cat}} = \frac{0 + (1.0 \cdot 30.0)}{0 + 1.0} = \mathbf{30.0}$
    - Sample 3 ($\text{"City\_A"}$): $\hat{x}_{3, \text{cat}} = \frac{10 + (1.0 \cdot 30.0)}{1 + 1.0} = \frac{40}{2} = \mathbf{20.0}$
    - Sample 4 ($\text{"City\_B"}$): $\hat{x}_{4, \text{cat}} = \frac{20 + (1.0 \cdot 30.0)}{1 + 1.0} = \frac{50}{2} = \mathbf{25.0}$
    $$\text{Encoded Data Matrix } X_{\text{encoded}} = \begin{bmatrix} 30.0 & 1.0 \\ 30.0 & 2.0 \\ 20.0 & 5.0 \\ 25.0 & 6.0 \end{bmatrix}$$

3. **Ordered Boosting: Permutation Residuals (Gradients)**

    - Sample 1 ($i=1$): Sub-model $M_0() = 30.0$
    $$g_1 = M_0(x_1) - y_1 = 30.0 - 10 = \mathbf{+20.0}, \quad h_1 = 1.0$$
    - Sample 2 ($i=2$): Sub-model $M_1(\{1\}) = \bar{y}_{\{1\}} = 10.0$
    $$g_2 = M_1(x_2) - y_2 = 10.0 - 20 = \mathbf{-10.0}, \quad h_2 = 1.0$$
    - Sample 3 ($i=3$): Sub-model $M_2(\{1,2\}) = \frac{10 + 20}{2} = 15.0$
    $$g_3 = M_2(x_3) - y_3 = 15.0 - 40 = \mathbf{-25.0}, \quad h_3 = 1.0$$
    - Sample 4 ($i=4$): Sub-model $M_3(\{1,2,3\}) = \frac{10 + 20 + 40}{3} = 23.33$
    $$g_4 = M_3(x_4) - y_4 = 23.33 - 50 = \mathbf{-26.67}, \quad h_4 = 1.0$$

4. **Oblivious (Symmetric) Tree Split Finding**

    Test single symmetric split rule across all nodes: $X_{\text{num}} \le 3.5$
    
    - Left Leaf ($L_1$, Samples 1 & 2 where $X_{\text{num}} \le 3.5$):
    $$\sum g_L = +20.0 - 10.0 = \mathbf{+10.0}, \quad \sum h_L = 1.0 + 1.0 = \mathbf{2.0}$$
    $$\text{Sim}_L = \frac{(+10.0)^2}{2.0 + 1.0} = \frac{100}{3.0} = \mathbf{33.33}$$
    $$w_L = -\frac{\sum g_L}{\sum h_L + \lambda} = -\frac{+10.0}{2.0 + 1.0} = \mathbf{-3.33}$$
    
    - Right Leaf ($L_2$, Samples 3 & 4 where $X_{\text{num}} > 3.5$):
    $$\sum g_R = -25.0 - 26.67 = \mathbf{-51.67}, \quad \sum h_R = 1.0 + 1.0 = \mathbf{2.0}$$
    $$\text{Sim}_R = \frac{(-51.67)^2}{2.0 + 1.0} = \frac{2669.79}{3.0} = \mathbf{889.93}$$
    $$w_R = -\frac{\sum g_R}{\sum h_R + \lambda} = -\frac{-51.67}{2.0 + 1.0} = \mathbf{+17.22}$$

5. **Prediction Update ($m=1$)**
    $$\hat{y}_i^{(1)} = \hat{y}_i^{(0)} + \eta \cdot w(x_i)$$

    |Sample $i$|True $y_i$​|$\hat y_​i^{(0)}$|​Leaf Assigned|Weight w|Updated Prediction $ \hat y_​i^{(1)}$|
    |---|---|---|---|---|---|
    |​1|10|30.0|Left ($L_1$)|$-3.33$|$30.0 + 0.5(-3.33) = \mathbf{28.34}$|
    |2|20|30.0|Left ($L_1$)|$-3.33$|$30.0 + 0.5(-3.33) = \mathbf{28.34}$|
    |3|40|30.0|Right ($L_2$)|$+17.22$|$30.0 + 0.5(+17.22) = \mathbf{38.61}$|
    |4|50|30.0|Right ($L_2$)|$+17.22$|$30.0 + 0.5(+17.22) = \mathbf{38.61}$|


# Python Code

In [7]:
import numpy as np


class CatBoostRegressorFromScratch:
    def __init__(self, n_estimators=3, learning_rate=0.5, reg_lambda=1.0, a=1.0):
        self.M = n_estimators
        self.eta = learning_rate
        self.reg_lambda = reg_lambda
        self.a = a
        self.global_prior = 0.0
        self.trees = []

    def _compute_ordered_target_stats(self, X_cat, y):
        N = len(X_cat)
        encoded_cat = np.zeros(N, dtype=np.float64)
        for i in range(N):
            past_mask = X_cat[:i] == X_cat[i]
            matching_y_sum = np.sum(y[:i][past_mask])
            matching_count = np.sum(past_mask)
            encoded_cat[i] = (matching_y_sum + self.a * self.global_prior) / (
                matching_count + self.a
            )
        return encoded_cat

    def _compute_residuals(self, y_hat, y):
        """Fixed: Residuals update dynamically based on current predictions y_hat"""
        return y_hat - y

    def _fit_oblivious_tree(self, X_full, g, h):
        N, n_features = X_full.shape
        best_gain = -1e9
        best_split = None

        root_sim = (np.sum(g) ** 2) / (np.sum(h) + self.reg_lambda)

        for f_idx in range(n_features):
            col = X_full[:, f_idx]
            thresholds = (
                np.sort(np.unique(col))[:-1] + np.sort(np.unique(col))[1:]
            ) / 2.0

            for t in thresholds:
                left_mask = col <= t
                right_mask = ~left_mask

                g_L, h_L = g[left_mask], h[left_mask]
                g_R, h_R = g[right_mask], h[right_mask]

                if len(g_L) == 0 or len(g_R) == 0:
                    continue

                sim_L = (np.sum(g_L) ** 2) / (np.sum(h_L) + self.reg_lambda)
                sim_R = (np.sum(g_R) ** 2) / (np.sum(h_R) + self.reg_lambda)
                gain = sim_L + sim_R - root_sim

                if gain > best_gain:
                    best_gain = gain
                    w_L = -np.sum(g_L) / (np.sum(h_L) + self.reg_lambda)
                    w_R = -np.sum(g_R) / (np.sum(h_R) + self.reg_lambda)
                    best_split = {
                        "feature_idx": f_idx,
                        "threshold": t,
                        "w_L": w_L,
                        "w_R": w_R,
                    }

        return best_split

    def fit(self, X_cat, X_num, y):
        self.global_prior = np.mean(y)
        y_hat = np.full_like(y, self.global_prior, dtype=np.float64)

        encoded_cat = self._compute_ordered_target_stats(X_cat, y)
        X_full = np.column_stack([encoded_cat, X_num])

        print(f"Target y: {y}")
        print(f"Initial Prior y_bar: {self.global_prior:.2f}\n")

        for m in range(self.M):
            # 1. Compute dynamic residuals: g_i = y_hat_i - y_i
            g = self._compute_residuals(y_hat, y)
            h = np.ones_like(g)

            # 2. Fit tree on current residual
            tree = self._fit_oblivious_tree(X_full, g, h)
            self.trees.append(tree)

            # 3. Update predictions
            mask = X_full[:, tree["feature_idx"]] <= tree["threshold"]
            w_all = np.where(mask, tree["w_L"], tree["w_R"])
            y_hat += self.eta * w_all

            print(f"--- Round {m+1} ---")
            print("Current Gradients (g_i):", np.round(g, 2))
            print(
                f"Split Rule: Feature {tree['feature_idx']} <= {tree['threshold']:.2f}"
            )
            print("Updated Predictions:", np.round(y_hat, 2))


if __name__ == "__main__":
    X_cat = np.array(["City_A", "City_B", "City_A", "City_B"])
    X_num = np.array([1.0, 2.0, 5.0, 6.0])
    y = np.array([10.0, 20.0, 40.0, 50.0])

    model = CatBoostRegressorFromScratch(
        n_estimators=20, learning_rate=0.2, reg_lambda=1.0, a=1.0
    )
    model.fit(X_cat, X_num, y)

Target y: [10. 20. 40. 50.]
Initial Prior y_bar: 30.00

--- Round 1 ---
Current Gradients (g_i): [ 20.  10. -10. -20.]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [28. 28. 32. 32.]
--- Round 2 ---
Current Gradients (g_i): [ 18.   8.  -8. -18.]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [26.27 26.27 33.73 33.73]
--- Round 3 ---
Current Gradients (g_i): [ 16.27   6.27  -6.27 -16.27]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [24.76 24.76 35.24 35.24]
--- Round 4 ---
Current Gradients (g_i): [ 14.76   4.76  -4.76 -14.76]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [23.46 23.46 36.54 36.54]
--- Round 5 ---
Current Gradients (g_i): [ 13.46   3.46  -3.46 -13.46]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [22.33 22.33 37.67 37.67]
--- Round 6 ---
Current Gradients (g_i): [ 12.33   2.33  -2.33 -12.33]
Split Rule: Feature 0 <= 27.50
Updated Predictions: [21.36 21.36 38.64 38.64]
--- Round 7 ---
Current Gradients (g_i): [ 11.36   1.36  -1.36 -11.36]
Spl